In [ ]:
import glob
import os
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import boto3
import matplotlib.pyplot as plt
import fabio
from tqdm import tqdm
import h5pyd
import zarr
from zarr.storage import FsspecStore
from numcodecs import Blosc
from zarr.codecs.gzip import GzipCodec
from zarr.codecs import BloscCodec, BloscCname
import fsspec
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow.fs import S3FileSystem
import tiledb

In [ ]:
os.environ['AWS_ACCESS_KEY_ID'] = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''
os.environ['BUCKET_NAME'] = 'scatterin-thesis'
os.environ['AWS_REGION'] = 'eu-north-1'
os.environ['AWS_S3_GATEWAY'] = 'http://s3.amazonaws.com'

os.environ['SN_PORT'] = '5101'
os.environ['HSDS_ENDPOINT'] = 'http://localhost:5101'
os.environ['HS_ENDPOINT'] = 'http://localhost:5101'
os.environ['LOG_LEVEL'] = 'INFO'
os.environ['HS_USERNAME'] = 'test_user1'
os.environ['HS_PASSWORD'] = 'test'

aws_opts = {
    "key":    os.environ["AWS_ACCESS_KEY_ID"],
    "secret": os.environ["AWS_SECRET_ACCESS_KEY"],
    "client_kwargs": {"region_name": os.environ["AWS_REGION"]},
}

In [ ]:
MAX_WORKERS = 32
BATCH = 16

In [ ]:
CBF_DIR = "../synthetic_images_v2"
cbf_files = sorted(glob.glob(os.path.join(CBF_DIR, "*.cbf")))
num_files = len(cbf_files)
if num_files == 0:
    raise RuntimeError(f"No .cbf files found in {CBF_DIR}")

sample = fabio.open(cbf_files[0]).data
height, width = sample.shape
dtype = sample.dtype

In [ ]:
def load_frame(path: str) -> np.ndarray:
    return fabio.open(path).data

# HDF5

In [ ]:
fname = "/home/test_user1/dummy5.h5"
with h5pyd.File(
      fname,
      "w",
      endpoint="http://localhost:5101",
      username="test_user1",
      password="test"
    ) as f:
    data = np.arange(100).reshape(10, 10)
    f.create_dataset("my_dummy", data=data)
print("Wrote dummy dataset →", fname)

### GZIP

In [ ]:
fname = "/hdf5/gzip/all_data.h5"

with h5pyd.File(
      fname,
      "w",
      endpoint="http://localhost:5101",
      username="test_user1",
      password="test"
    ) as f:
    dset = f.create_dataset(
        "data",
         shape=(num_files, height, width),
        dtype=dtype,
        compression="gzip",
        chunks=(1, height, width),
    )

    executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)

    for start in tqdm(range(0, num_files, 64), desc="Writing batches"):
        end = min(start + 64, num_files)
        batch_paths = cbf_files[start:end]

        futures = [executor.submit(load_frame, p) for p in batch_paths]

        stack = np.empty((end - start, height, width), dtype=dtype)
        for i, fut in enumerate(futures):
            stack[i] = fut.result()

        dset[start:end, :, :] = stack

    executor.shutdown()

### LZ4

In [ ]:
fname = "/hdf5/lz4/all_data.h5"

with h5pyd.File(
      fname,
      "w",
      endpoint="http://localhost:5101",
      username="test_user1",
      password="test"
    ) as f:
    dset = f.create_dataset(
        "data",
         shape=(num_files, height, width),
        dtype=dtype,
        compression="lz4",
        chunks=(1, height, width),
    )

    executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)

    for start in tqdm(range(0, num_files, 64), desc="Writing batches"):
        end = min(start + 64, num_files)
        batch_paths = cbf_files[start:end]

        futures = [executor.submit(load_frame, p) for p in batch_paths]

        stack = np.empty((end - start, height, width), dtype=dtype)
        for i, fut in enumerate(futures):
            stack[i] = fut.result()

        dset[start:end, :, :] = stack

    executor.shutdown()

### ZSTD

In [ ]:
fname = "/hdf5/zstd/all_data.h5"

with h5pyd.File(
      fname,
      "w",
      endpoint="http://localhost:5101",
      username="test_user1",
      password="test"
    ) as f:
    dset = f.create_dataset(
        "data",
         shape=(num_files, height, width),
        dtype=dtype,
        compression="zstd",
        chunks=(1, height, width),
    )

    executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)

    for start in tqdm(range(0, num_files, 64), desc="Writing batches"):
        end = min(start + 64, num_files)
        batch_paths = cbf_files[start:end]

        futures = [executor.submit(load_frame, p) for p in batch_paths]

        stack = np.empty((end - start, height, width), dtype=dtype)
        for i, fut in enumerate(futures):
            stack[i] = fut.result()

        dset[start:end, :, :] = stack

    executor.shutdown()

# ZARR

### GZIP

In [ ]:
store_url = f"s3://scatterin-thesis/zarr/gzip.zarr"
store = FsspecStore.from_url(store_url, storage_options=aws_opts)
root  = zarr.group(store=store, overwrite=True)


dset = root.create_array(
    name="data",
    shape=(num_files, height, width),
    chunks=(1, height, width),
    dtype=dtype,
    compressor=GzipCodec(level=3)
)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for start in tqdm(range(0, num_files, BATCH), desc=f"Writing gzip"):
        end = min(start + BATCH, num_files)
        paths = cbf_files[start:end]

        frames = list(executor.map(load_frame, paths))  
        chunk  = np.stack(frames, axis=0)

        dset[start:end, :, :] = chunk

print(f"✅ Done gzip.zarr → {store_url}")

### ZSTD

In [ ]:
store_url_zstd = f"s3://scatterin-thesis/zarr/zstd.zarr"
store_zstd = FsspecStore.from_url(store_url_zstd, storage_options=aws_opts)
root_zstd = zarr.group(store=store_zstd, overwrite=True)

dset_zstd = root_zstd.create_array(
    name="data",
    shape=(num_files, height, width),
    chunks=(1, height, width),
    dtype=dtype,
    compressor=zarr.codecs.ZstdCodec(level=3)
)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for start in tqdm(range(0, num_files, BATCH), desc=f"Writing zstd"):
        end = min(start + BATCH, num_files)
        paths = cbf_files[start:end]

        frames = list(executor.map(load_frame, paths))
        chunk = np.stack(frames, axis=0)

        dset_zstd[start:end, :, :] = chunk

print(f"✅ Done zstd.zarr → {store_url_zstd}")

### LZ4

In [ ]:

store_url_lz4 = f"s3://scatterin-thesis/zarr/lz4.zarr"
store_lz4 = FsspecStore.from_url(store_url_lz4, storage_options=aws_opts)
root_lz4 = zarr.group(store=store_lz4, overwrite=True)

lz4_codec = BloscCodec(cname=BloscCname.lz4, clevel=3)

dset_lz4 = root_lz4.create_array(
    name="data",
    shape=(num_files, height, width),
    chunks=(1, height, width),
    dtype=dtype,
    compressor=lz4_codec
)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for start in tqdm(range(0, num_files, BATCH), desc=f"Writing lz4"):
        end = min(start + BATCH, num_files)
        paths = cbf_files[start:end]

        frames = list(executor.map(load_frame, paths))
        chunk = np.stack(frames, axis=0)

        dset_lz4[start:end, :, :] = chunk

print(f"✅ Done lz4.zarr → {store_url_lz4}")

# Parquet

In [ ]:
fs = S3FileSystem(
    region=os.environ["AWS_REGION"],
    access_key=os.environ["AWS_ACCESS_KEY_ID"],
    secret_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

inner = pa.from_numpy_dtype(dtype)
pixels = pa.list_(inner, width)

fields = [(f"row_{i}", pixels) for i in range(height)]
schema = pa.schema(fields)

### Inspect

In [ ]:
parquet_path = "scatterin-thesis/parquet/lz4.parquet"

tbl = pq.read_table(parquet_path, filesystem=fs)
print(f"Frames (rows): {tbl.num_rows},  Columns: {tbl.num_columns}")

names  = tbl.schema.names
frame0 = np.stack([tbl[name][0].as_py() for name in names], axis=0)
print("Frame0 shape:", frame0.shape, "dtype:", frame0.dtype)

plt.figure(figsize=(6,6))
plt.imshow(frame0)
plt.colorbar()
plt.title("Frame 0 Preview")
plt.show()

### GZIP

In [ ]:
out = fs.open_output_stream("scatterin-thesis/parquet/gzip.parquet")
writer = pq.ParquetWriter(out, schema=schema, compression="GZIP", use_dictionary=False)

for idx, path in enumerate(tqdm(cbf_files, desc="Writing flattened Parquet")):
    frame = load_frame(path)
    
    arrays = []
    names  = []
    for i in range(height):
        row_pixels = frame[i].tolist()
        arr = pa.array([row_pixels], type=pixels)
        arrays.append(arr)
        names.append(f"row_{i}")

    table = pa.Table.from_arrays(arrays, schema=schema)
    writer.write_table(table)

writer.close()
print("✅ Done → s3://scatterin-thesis/parquet/gzip_flat_columns.parquet")

### LZ4

In [ ]:
out = fs.open_output_stream("scatterin-thesis/parquet/lz4.parquet")
writer = pq.ParquetWriter(out, schema=schema, compression="LZ4", use_dictionary=False)

for idx, path in enumerate(tqdm(cbf_files, desc="Writing LZ4 Parquet")):
    frame = load_frame(path)
    
    arrays = []
    for i in range(height):
        row_pixels = frame[i].tolist()
        arr = pa.array([row_pixels], type=pixels)
        arrays.append(arr)

    table = pa.Table.from_arrays(arrays, schema=schema)
    writer.write_table(table)

writer.close()
print("✅ Done → s3://scatterin-thesis/parquet/lz4.parquet")

# TileDB

In [ ]:
cfg = {
    "vfs.s3.region":               aws_opts["client_kwargs"]["region_name"],
    "vfs.s3.aws_access_key_id":    aws_opts["key"],
    "vfs.s3.aws_secret_access_key":aws_opts["secret"],
    "vfs.s3.scheme":               "https",
    "vfs.s3.use_virtual_addressing":"true",
}
ctx = tiledb.Ctx(tiledb.Config(cfg))

dom = tiledb.Domain(
    tiledb.Dim(name="file", domain=(0, num_files - 1), tile=(1,), dtype=np.uint32),
    tiledb.Dim(name="y",    domain=(0, height - 1),   tile=(height,), dtype=np.uint32),
    tiledb.Dim(name="x",    domain=(0, width - 1),    tile=(width,),  dtype=np.uint32),
    ctx=ctx
)

In [ ]:
def write_chunk(start, uri):
    end = min(start + BATCH, num_files)
    paths = cbf_files[start:end]
    frames = [fabio.open(p).data for p in paths]
    chunk  = np.stack(frames, axis=0)
    with tiledb.DenseArray(uri, mode='w', ctx=ctx) as A:
        A[start:end, :, :] = chunk

### GZIP

In [ ]:
uri = "s3://scatterin-thesis/tiledb/gzip.tdb"

attr = tiledb.Attr(
    name="data",
    dtype=dtype,
    filters=tiledb.FilterList([tiledb.GzipFilter(level=3)]),
    ctx=ctx
)

schema = tiledb.ArraySchema(
    domain=dom,
    sparse=False,
    attrs=[attr],
    ctx=ctx
)

tiledb.DenseArray.create(uri, schema, ctx=ctx)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    list(tqdm(
        executor.map(lambda start: write_chunk(start, uri), range(0, num_files, BATCH)),
        total=(num_files + BATCH - 1) // BATCH,
        desc="Writing gzip TileDB"
    ))

### LZ4

In [ ]:
uri_lz4 = "s3://scatterin-thesis/tiledb/lz4.tdb"
attr_lz4 = tiledb.Attr(
    name="data",
    dtype=dtype,
    filters=tiledb.FilterList([tiledb.LZ4Filter(level=3)]),
    ctx=ctx
)
schema_lz4 = tiledb.ArraySchema(
    domain=dom,
    sparse=False,
    attrs=[attr_lz4],
    ctx=ctx
)
tiledb.DenseArray.create(uri_lz4, schema_lz4, ctx=ctx)

with ThreadPoolExecutor(max_workers=MAX_WORKERS // 8) as executor:
    list(tqdm(
        executor.map(lambda start: write_chunk(start, uri_lz4), range(0, num_files, BATCH)),
        total=(num_files + BATCH - 1) // BATCH,
        desc="Writing lz4 TileDB"
    ))

### ZSTD

In [ ]:
uri_zstd = "s3://scatterin-thesis/tiledb/zstd.tdb"
attr_zstd = tiledb.Attr(
    name="data",
    dtype=dtype,
    filters=tiledb.FilterList([tiledb.ZstdFilter(level=3)]),
    ctx=ctx
)
schema_zstd = tiledb.ArraySchema(
    domain=dom,
    sparse=False,
    attrs=[attr_zstd],
    ctx=ctx
)
tiledb.DenseArray.create(uri_zstd, schema_zstd, ctx=ctx)

with ThreadPoolExecutor(max_workers=MAX_WORKERS // 8) as executor:
    list(tqdm(
        executor.map(lambda start: write_chunk(start, uri_zstd), range(0, num_files, BATCH)),
        total=(num_files + BATCH - 1) // BATCH,
        desc="Writing zstd TileDB"
    ))

# ROOT

In [ ]:
os.environ['C_INCLUDE_PATH'] = '/usr/include'
os.environ['CPLUS_INCLUDE_PATH'] = '/usr/include'

import ROOT
import uproot
from array import array

max_pixels = height * width


### View

In [ ]:
store_url = "s3://scatterin-thesis/root/lz4.root"

ufile = uproot.open(store_url)
print("Contents:", ufile.keys())

tree   = ufile["img_tree"]
branch = tree["data"]

arr1d = branch.array(entry_start=12, entry_stop=13, library="np")[0]
print("Raw length:", arr1d.shape)

frame0 = arr1d.reshape((height, width))

print("Frame 0 shape:", frame0.shape)
print(frame0)

plt.figure(figsize=(8, 8))
plt.imshow(frame0, cmap="viridis")
plt.colorbar()
plt.title("Frame 0 Preview")
plt.show()

### GZIP

In [ ]:
f = ROOT.TFile("gzip.root", "RECREATE")

f.SetCompressionAlgorithm(ROOT.RCompressionSetting.EAlgorithm.kZLIB)
f.SetCompressionLevel(3)

tree  = ROOT.TTree("img_tree", "CBF Image Frames")
c_data = array('H', [0] * max_pixels)
tree.Branch("data", c_data, f"data[{max_pixels}]/s")

buffer_view = np.frombuffer(c_data, dtype=np.uint16)

for path in tqdm(cbf_files, desc="Writing TTree → gzip"):
    arr = fabio.open(path).data.astype(np.uint16)
    flat = arr.ravel()
    buffer_view[:] = flat
    tree.Fill()

tree.Write()
f.Close()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
    region_name=os.environ['AWS_REGION']
)

local_path = "gzip.root"
bucket = "scatterin-thesis"
s3_key = "root/gzip.root"

s3.upload_file(local_path, bucket, s3_key)

### LZ4

In [ ]:
f = ROOT.TFile("lz4.root", "RECREATE")

f.SetCompressionAlgorithm(ROOT.RCompressionSetting.EAlgorithm.kLZ4)
f.SetCompressionLevel(ROOT.RCompressionSetting.ELevel.kDefaultLZ4)

tree  = ROOT.TTree("img_tree", "CBF Image Frames")
c_data = array('H', [0] * max_pixels)
tree.Branch("data", c_data, f"data[{max_pixels}]/s")

buffer_view = np.frombuffer(c_data, dtype=np.uint16)

for path in tqdm(cbf_files, desc="Writing TTree → lz4"):
    arr = fabio.open(path).data.astype(np.uint16)
    flat = arr.ravel()
    buffer_view[:] = flat
    tree.Fill()

tree.Write()
f.Close()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
    region_name=os.environ['AWS_REGION']
)

local_path = "lz4.root"
bucket = "scatterin-thesis"
s3_key = "root/lz4.root"

s3.upload_file(local_path, bucket, s3_key)

### ZSTD

In [ ]:
f = ROOT.TFile("zstd.root", "RECREATE")

f.SetCompressionAlgorithm(ROOT.RCompressionSetting.EAlgorithm.kZSTD);
f.SetCompressionLevel(ROOT.RCompressionSetting.ELevel.kDefaultZSTD);

tree  = ROOT.TTree("img_tree", "CBF Image Frames")
c_data = array('H', [0] * max_pixels)
tree.Branch("data", c_data, f"data[{max_pixels}]/s")

buffer_view = np.frombuffer(c_data, dtype=np.uint16)

for path in tqdm(cbf_files, desc="Writing TTree → zstd"):
    arr = fabio.open(path).data.astype(np.uint16)
    flat = arr.ravel()
    buffer_view[:] = flat
    tree.Fill()

tree.Write()
f.Close()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
    region_name=os.environ['AWS_REGION']
)

local_path = "zstd.root"
bucket = "scatterin-thesis"
s3_key = "root/zstd.root"

s3.upload_file(local_path, bucket, s3_key)